# GrC Image Matting on CBIS-DDSM - Layers 2–4 and the Feature Diagnostic

Template matching, histogram pooling, and SVM training - plus the diagnostic investigation that is the main finding of this phase.

Writes to Drive: `svm_layer4.joblib`, `svm_scaler.joblib`, `svm_results.json`, `layer3_hist.npz`.

## State restoration

Reloads the Layer 1 maps, templates and split from Drive.

In [ ]:
# Imports and config
!pip install -q opencv-python scikit-learn scipy tqdm joblib

import os, json, random, warnings, time
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
import joblib
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = "/content/drive/MyDrive/cbis_ddsm_grc_project"
CONFIG = json.load(open(os.path.join(DRIVE_ROOT, "config.json")))
globals().update(CONFIG)
SEED = CONFIG["SEED"]
random.seed(SEED); np.random.seed(SEED)

MAX_TRAIN_PATCHES_PER_IMG = 300
MAX_GRID, MAX_FIT = 8000, 20000
print("Config loaded.")

Mounted at /content/drive
Config loaded.


In [ ]:
final_df_loaded = pd.read_csv(os.path.join(RESULTS_DIR, "final_df_loaded.csv"))
metadata_df     = pd.read_csv(os.path.join(PREPROCESSED_DIR, "preprocessing_metadata.csv"))
roi_arch    = np.load(os.path.join(PREPROCESSED_DIR, "rois.npz"))
trimap_arch = np.load(os.path.join(TRIMAPS_DIR, "trimaps.npz"))
gtmask_arch = np.load(os.path.join(PREPROCESSED_DIR, "gt_masks.npz"))

lesion_of = dict(zip(metadata_df['image_id'], metadata_df['lesion_key']))
class_of  = dict(zip(final_df_loaded['image_id'], final_df_loaded['class']))
preprocessed_results = {}
for iid in roi_arch.files:
    preprocessed_results[iid] = {
        'image_id': iid, 'roi': roi_arch[iid],
        'trimap': trimap_arch[iid] if iid in trimap_arch.files else None,
        'gt_mask': gtmask_arch[iid] if iid in gtmask_arch.files else None,
        'lesion_key': lesion_of.get(iid), 'class': class_of.get(iid),
    }

templates = np.load(os.path.join(MODELS_DIR, "kmeans_templates.npy"))
K_TEMPLATES = templates.shape[0]
tmeta = json.load(open(os.path.join(MODELS_DIR, "template_meta.json")))
assert tmeta['K'] == K_TEMPLATES

sp = json.load(open(os.path.join(RESULTS_DIR, "train_test_split.json")))
train_ids, test_ids = set(sp['train_ids']), set(sp['test_ids'])

layer1_ms_maps = {}
for ps in PATCH_SIZES:
    a = np.load(os.path.join(OUTPUTS_DIR, f"layer1_ms_lipw_{ps}x{ps}.npz"))
    for iid in a.files:
        if iid not in layer1_ms_maps:
            layer1_ms_maps[iid] = {
                'lipw_stack': np.zeros((*a[iid].shape, len(PATCH_SIZES)), np.float32),
                'cls': class_of.get(iid, 'UNKNOWN')}
        layer1_ms_maps[iid]['lipw_stack'][:, :, PATCH_SIZES.index(ps)] = a[iid]

print(f"Restored: {len(preprocessed_results)} images, K={K_TEMPLATES}, "
      f"{len(layer1_ms_maps)} Layer 1 stacks")
assert not (set(sp['train_lesions']) & set(sp['test_lesions'])), "lesion leakage"
print("No lesion-level leakage.")

Restored: 537 images, K=16, 537 Layer 1 stacks
No lesion-level leakage.


## Sections 9 & 10 - Layer 2 matching and Layer 3 histograms

Same signed matcher and integral-image histogram as the MIAS phase.

In [ ]:
#  Layer 2 matcher and Layer 3 histogram 
def match_templates_signed(lipw_map, templates):
    h, w = lipw_map.shape
    K    = len(templates)
    ps   = templates.shape[1]
    half = ps // 2

    padded  = np.pad(lipw_map, half, mode='reflect').astype(np.float32)
    patches = sliding_window_view(padded, (ps, ps))
    pf      = patches.reshape(h * w, ps * ps).astype(np.float32)

    T  = templates.reshape(K, ps * ps).astype(np.float32)
    Tc = T - T.mean(axis=1, keepdims=True)
    W  = np.sign(Tc)
    W[W == 0] = 1.0

    Pc = pf - pf.mean(axis=1, keepdims=True)

    num   = Pc @ W.T
    denom = (np.linalg.norm(Pc, axis=1, keepdims=True) *
             np.linalg.norm(W, axis=1)[None, :] + 1e-8)
    corr  = num / denom
    score = (corr + 1.0) * 0.5
    return score.reshape(h, w, K).astype(np.float32)


def match_templates_old(lipw_map, templates):
    h, w = lipw_map.shape
    K    = len(templates)
    ps   = templates.shape[1]
    half = ps // 2

    padded  = np.pad(lipw_map, half, mode='reflect').astype(np.float32)
    patches = sliding_window_view(padded, (ps, ps))
    pf      = patches.reshape(h * w, ps * ps)
    tf      = templates.reshape(K, ps * ps)
    out     = np.zeros((h * w, K), dtype=np.float32)

    for k in range(K):
        t = tf[k]
        high = (t >= 0.7)
        low  = (t <= 0.3)
        med  = ~high & ~low
        sat  = np.zeros((h * w, ps * ps), dtype=np.float32)
        if high.any():
            sat[:, high] = pf[:, high]
        if low.any():
            sat[:, low] = 1.0 - pf[:, low]
        if med.any():
            sat[:, med] = 1.0 - np.abs(pf[:, med] - 0.5) * 2.0
        b = np.maximum(0, sat[:, :-1] + sat[:, 1:] - 1)
        out[:, k] = b.mean(axis=1)

    return out.reshape(h, w, K)

print("Both matchers defined (old is diagnostic only).")


def compute_histogram_map_fast(score_map, window_size=7, stride=1):
    h, w, k = score_map.shape

    # boxFilter computes the MEAN over the window; multiply by window area
    # to get the SUM, matching the original integral-image sum exactly.
    win_area = float(window_size * window_size)
    summed = np.empty_like(score_map, dtype=np.float32)
    for c in range(k):
        summed[:, :, c] = cv2.boxFilter(
            score_map[:, :, c], ddepth=-1, ksize=(window_size, window_size),
            borderType=cv2.BORDER_REFLECT, normalize=True
        ) * win_area

    if stride > 1:
        summed = summed[::stride, ::stride, :]

    totals = summed.sum(axis=2, keepdims=True)
    out = np.where(totals > 1e-8, summed / totals, 1.0 / k).astype(np.float32)
    return out

print("compute_histogram_map_fast replaced with a vectorized version.")
print("Same math, no Python-level pixel loop -- much faster, lower peak RAM.")

Both matchers defined (old is diagnostic only).
compute_histogram_map_fast replaced with a vectorized version.
Same math, no Python-level pixel loop -- much faster, lower peak RAM.


In [ ]:
# Helper function to align trimap
def align_to(map2d_shape, arr, nearest=True):
    if arr is None:
        return None
    if arr.shape[:2] == map2d_shape:
        return arr
    interp = cv2.INTER_NEAREST if nearest else cv2.INTER_LINEAR
    return cv2.resize(arr, (map2d_shape[1], map2d_shape[0]), interpolation=interp)

scale_idx = PATCH_SIZES.index(PATCH_SINGLE)
print(f"Layer 2 runs on the {PATCH_SINGLE}x{PATCH_SINGLE} scale.")

Layer 2 runs on the 5x5 scale.


In [ ]:
# Batch Layer 2 matcher
print("Building Layer 2 score maps...\n")
layer2_maps = {}
t0 = time.time()
for _, row in tqdm(final_df_loaded.iterrows(), total=len(final_df_loaded), desc="Layer 2"):
    iid = row['image_id']
    if iid not in layer1_ms_maps:
        continue
    lipw = layer1_ms_maps[iid]['lipw_stack'][:, :, scale_idx]
    layer2_maps[iid] = {'score_map': match_templates_signed(lipw, templates),
                        'cls': row['class']}
print(f"\nDone: {len(layer2_maps)} maps in {(time.time()-t0)/60:.1f} min")
np.savez_compressed(os.path.join(OUTPUTS_DIR, "layer2_score_maps.npz"),
                    **{k: v['score_map'] for k, v in layer2_maps.items()})
print("Saved layer2_score_maps.npz")

Building Layer 2 score maps...



Layer 2: 100%|██████████| 537/537 [00:04<00:00, 116.98it/s]



Done: 537 maps in 0.1 min
Saved layer2_score_maps.npz


In [ ]:
# Layer 3 histogram
print("Rebuilding Layer 3 histograms from the already-loaded Layer 2 maps...")
layer3_maps = {}
for _, row in tqdm(final_df_loaded.iterrows(), total=len(final_df_loaded), desc="Layer 3"):
    iid = row['image_id']
    if iid not in layer2_maps:
        continue
    layer3_maps[iid] = {'hist_map': compute_histogram_map_fast(
        layer2_maps[iid]['score_map'], HISTOGRAM_WINDOW, HISTOGRAM_STRIDE),
        'cls': row['class']}

print(f"Rebuilt {len(layer3_maps)} Layer 3 maps.")

LOCAL_TMP = "/content/layer3_histogram_maps_tmp.npz"
DRIVE_DEST = os.path.join(OUTPUTS_DIR, "layer3_histogram_maps.npz")

print("Saving to local disk first...")
np.savez_compressed(LOCAL_TMP, **{k: v['hist_map'] for k, v in layer3_maps.items()})

print("Verifying local file...")
local_check = np.load(LOCAL_TMP)
assert len(local_check.files) == len(layer3_maps), "Local save is incomplete!"
local_check.close()
print(f"Local file verified OK ({len(layer3_maps)} entries, "
      f"{os.path.getsize(LOCAL_TMP)/1e9:.2f} GB).")

print("Copying finished file to Drive...")
import shutil
shutil.copy(LOCAL_TMP, DRIVE_DEST)

print("Verifying Drive copy...")
drive_check = np.load(DRIVE_DEST)
assert len(drive_check.files) == len(layer3_maps), "Drive copy is incomplete!"
drive_check.close()
print(f"Drive copy verified OK. Safe to trust layer3_histogram_maps.npz now.")

# Same sanity check as before
sample_iid = next(iter(layer3_maps))
assert layer3_maps[sample_iid]['hist_map'].shape[2] == K_TEMPLATES
print(f"Confirmed histogram width matches current K_TEMPLATES ({K_TEMPLATES}).")

import gc
del layer2_maps
gc.collect()
print("Freed layer2_maps from memory (not needed downstream).")

print("Ready for Cell 8 (build_feature_table) or the diagnostic cell.")

Rebuilding Layer 3 histograms from the already-loaded Layer 2 maps...


Layer 3: 100%|██████████| 537/537 [00:01<00:00, 310.79it/s]


Rebuilt 537 Layer 3 maps.
Saving to local disk first...
Verifying local file...
Local file verified OK (537 entries, 0.21 GB).
Copying finished file to Drive...
Verifying Drive copy...
Drive copy verified OK. Safe to trust layer3_histogram_maps.npz now.
Confirmed histogram width matches current K_TEMPLATES (16).
Freed layer2_maps from memory (not needed downstream).
Ready for Cell 8 (build_feature_table) or the diagnostic cell.


In [ ]:
# Extends the single intensity feature to THREE scales, mirroring
# Modification 4's multi-scale texture idea but applied to brightness 
INTENSITY_SCALES = [5, 9, 15]  # small, medium, wider local neighbourhood

def local_intensity_maps(roi, scales=INTENSITY_SCALES):

    """Returns a list of local-mean-brightness maps, one per scale."""

    return [cv2.boxFilter(roi.astype(np.float32), -1, (w, w),
                          borderType=cv2.BORDER_REFLECT) for w in scales]

def build_feature_table(layer3_maps, preprocessed_results, df,
                        id_filter, max_per_img=MAX_TRAIN_PATCHES_PER_IMG):
    X, y, groups = [], [], []
    rng = np.random.default_rng(SEED)

    for _, row in df.iterrows():
        iid, cls = row['image_id'], row['class']
        if cls == 'NORM':
            continue
        if iid not in id_filter or iid not in layer3_maps:
            continue

        tm  = preprocessed_results[iid]['trimap']
        roi = preprocessed_results[iid]['roi']
        if tm is None or roi is None:
            continue

        hm = layer3_maps[iid]['hist_map']
        hh, ww = hm.shape[:2]
        if tm.shape != (hh, ww):
            tm = cv2.resize(tm, (ww, hh), interpolation=cv2.INTER_NEAREST)
        if roi.shape != (hh, ww):
            roi = cv2.resize(roi, (ww, hh), interpolation=cv2.INTER_LINEAR)

        ims = local_intensity_maps(roi)  

        pos_mass = np.argwhere(tm == 255)
        pos_bg   = np.argwhere(tm == 0)
        if len(pos_mass) == 0 or len(pos_bg) == 0:
            continue

        n = min(len(pos_mass), len(pos_bg), max_per_img)
        for pos, label in [(pos_mass, 1), (pos_bg, 0)]:
            if len(pos) > n:
                pos = pos[rng.choice(len(pos), n, replace=False)]
            for (i, j) in pos:
                intensity_feats = [im[i, j] for im in ims]
                feat = np.concatenate([hm[i, j, :], intensity_feats])
                X.append(feat)
                y.append(label)
                groups.append(iid)

    return np.array(X, dtype=np.float32), np.array(y), np.array(groups)

def predict_prob_map(hist_map, roi, svm, scaler, batch=20000):
    hh, ww, k = hist_map.shape
    if roi.shape != (hh, ww):
        roi = cv2.resize(roi, (ww, hh), interpolation=cv2.INTER_LINEAR)
    ims = local_intensity_maps(roi)

    flat_hist = hist_map.reshape(-1, k)
    flat_int  = np.stack([im.reshape(-1) for im in ims], axis=1)  # (H*W, n_scales)
    flat = np.concatenate([flat_hist, flat_int], axis=1)

    out = np.zeros(len(flat), dtype=np.float32)
    for s in range(0, len(flat), batch):
        chunk = scaler.transform(flat[s:s+batch])
        out[s:s+batch] = svm.predict_proba(chunk)[:, 1]
    return out.reshape(hh, ww).astype(np.float32)

print(f"build_feature_table and predict_prob_map updated: {len(INTENSITY_SCALES)} "
      f"intensity scales {INTENSITY_SCALES} instead of 1.")

X_train, y_train, g_train = build_feature_table(
    layer3_maps, preprocessed_results, final_df_loaded, train_ids)
X_test, y_test, g_test = build_feature_table(
    layer3_maps, preprocessed_results, final_df_loaded, test_ids)

print(f"Train: X={X_train.shape}  mass={int(y_train.sum())}  bg={int((y_train==0).sum())}")
print(f"Test : X={X_test.shape}   mass={int(y_test.sum())}   bg={int((y_test==0).sum())}")
assert X_train.shape[1] == K_TEMPLATES + len(INTENSITY_SCALES)
print(f"Feature width confirmed: {K_TEMPLATES} + {len(INTENSITY_SCALES)} = "
      f"{K_TEMPLATES + len(INTENSITY_SCALES)}")

build_feature_table and predict_prob_map updated: 3 intensity scales [5, 9, 15] instead of 1.
Train: X=(87538, 19)  mass=43769  bg=43769
Test : X=(33314, 19)   mass=16657   bg=16657
Feature width confirmed: 16 + 3 = 19


In [ ]:
# Checks whether the histogram features themselves separate by class, to tell apart "genuinely weak signal" from "alignment bug"
mass_mean = X_train[y_train == 1].mean(axis=0)
bg_mean   = X_train[y_train == 0].mean(axis=0)
diff      = mass_mean - bg_mean

print("Dim   Mass mean   BG mean    Diff")
print("-" * 40)
for i in range(len(diff)):
    print(f"T{i:<4} {mass_mean[i]:.4f}     {bg_mean[i]:.4f}    {diff[i]:+.4f}")

print(f"\nMax abs diff across dims: {np.abs(diff).max():.4f}")
print(f"Mean abs diff across dims: {np.abs(diff).mean():.4f}")

print(f"\nFor reference, Notebook 4's discrimination score at K=16 was 0.1589")
print("(measured differently -- per-image positive-diff aggregate, not this")
print("direct per-dimension X_train mean comparison -- but should be in the")
print("same ballpark if features are consistent end-to-end)")

from scipy.spatial.distance import cdist
sample_idx = np.random.default_rng(SEED).choice(len(X_test), min(2000, len(X_test)), replace=False)
Xs, ys = X_test[sample_idx], y_test[sample_idx]
d_mass = cdist(Xs, mass_mean.reshape(1, -1)).ravel()
d_bg   = cdist(Xs, bg_mean.reshape(1, -1)).ravel()
pred   = (d_mass < d_bg).astype(int)
acc    = (pred == ys).mean()
print(f"\nNearest-centroid baseline accuracy on test sample: {acc:.4f}")
print("(if this is also ~0.50, the problem is upstream of the SVM entirely --")
print(" in the features/labels themselves, not the classifier or its tuning)")

Dim   Mass mean   BG mean    Diff
----------------------------------------
T0    0.0625     0.0620    +0.0006
T1    0.0626     0.0624    +0.0002
T2    0.0633     0.0623    +0.0011
T3    0.0621     0.0622    -0.0000
T4    0.0631     0.0621    +0.0010
T5    0.0634     0.0628    +0.0007
T6    0.0626     0.0628    -0.0002
T7    0.0631     0.0632    -0.0001
T8    0.0619     0.0619    -0.0001
T9    0.0621     0.0620    +0.0000
T10   0.0627     0.0632    -0.0005
T11   0.0619     0.0630    -0.0012
T12   0.0634     0.0632    +0.0002
T13   0.0615     0.0622    -0.0007
T14   0.0621     0.0623    -0.0002
T15   0.0616     0.0625    -0.0008
T16   0.6109     0.5855    +0.0253
T17   0.6097     0.5855    +0.0242
T18   0.6085     0.5853    +0.0232

Max abs diff across dims: 0.0253
Mean abs diff across dims: 0.0042

For reference, Notebook 4's discrimination score at K=16 was 0.1589
(measured differently -- per-image positive-diff aggregate, not this
direct per-dimension X_train mean comparison -- but sh

## Section 11 — Layer 4 SVM, and why intensity features were added

**This section is the main research finding of the CBIS-DDSM phase.** Three things
happen here, in order.

**1. The classifier collapsed to majority-class prediction.** Mass precision and
recall were both exactly 0.0000 behind an apparently preferable 0.6330 accuracy.
The cause was per-image sampling caps applied independently per class: the confident
foreground is only about 0.3% of a crop, so the background cap was always reached and
the mass cap almost never was. This is fixed with per-image balanced sampling,
`class_weight='balanced'`, and balanced accuracy as the scoring metric.

**2. The result was still near chance.** A raw feature-level diagnostic
explains why:

| Feature | Mass mean | Background mean | Difference |
| --- | --- | --- | --- |
| Local texture variance (5×5) | 0.022862 | 0.022907 | −0.000045 (ratio 0.998) |
| Raw pixel intensity | 0.5808 – 0.6039 | 0.5509 – 0.5782 | +0.026 – +0.030 |

The texture channel that Layers 1–3 are built on carries essentially no
mass-versus-background signal in this dataset. Intensity does.

CBIS-DDSM already has ~5× MIAS's images and performed *worse*. 
A nearest-centroid classifier - the simplest possible, immune to overfitting - scored
0.4905, below chance. Nearest-centroid, RBF-SVM and XGBoost all converged into the
same 53–56% band, which is the signature of a feature ceiling rather than a
classifier or sample-size problem. Crop tightening moved nothing (0.4905 → 0.5110).

**3. Local intensity was added to the Layer 4 input.** Three intensity scales (5/9/15 px)
extend the feature vector to 19 dimensions and lift test accuracy to 0.5593. XGBoost
importances show the widest window carries by far the most weight, which sharpens the
finding.

In [ ]:
scaler = StandardScaler().fit(X_train)
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

rng = np.random.default_rng(SEED)
gi = rng.choice(len(X_train_s), min(MAX_GRID, len(X_train_s)), replace=False)
print(f"Grid search on {len(gi)} samples...")

grid = GridSearchCV(
    SVC(kernel='rbf', probability=True, random_state=SEED, class_weight='balanced'),
    {'C': SVM_C_VALUES, 'gamma': SVM_GAMMA_VALUES},
    cv=3, scoring='balanced_accuracy', n_jobs=1, verbose=1)
grid.fit(X_train_s[gi], y_train[gi])
print(f"Best: {grid.best_params_}  CV balanced acc {grid.best_score_:.4f}")

fi = rng.choice(len(X_train_s), min(MAX_FIT, len(X_train_s)), replace=False)
svm = SVC(kernel='rbf', probability=True, random_state=SEED,
          class_weight='balanced', **grid.best_params_)
svm.fit(X_train_s[fi], y_train[fi])
test_acc = accuracy_score(y_test, svm.predict(X_test_s))
print(f"\nTest accuracy: {test_acc:.4f}\n")
print(classification_report(y_test, svm.predict(X_test_s),
                            target_names=['background', 'mass'], digits=4))

joblib.dump(svm, os.path.join(MODELS_DIR, "svm_layer4.joblib"))
joblib.dump(scaler, os.path.join(MODELS_DIR, "svm_scaler.joblib"))
json.dump({'best_params': grid.best_params_, 'best_cv_balanced_acc': float(grid.best_score_),
           'test_acc': float(test_acc),
           'n_train': int(len(fi)), 'n_test': int(len(X_test_s))},
          open(os.path.join(RESULTS_DIR, "svm_results.json"), 'w'), indent=2)
print("Saved SVM, scaler, svm_results.json")

Grid search on 8000 samples...
Fitting 3 folds for each of 6 candidates, totalling 18 fits
Best: {'C': 10, 'gamma': 'auto'}  CV balanced acc 0.5516

Test accuracy: 0.5593

              precision    recall  f1-score   support

  background     0.5817    0.4222    0.4893     16657
        mass     0.5465    0.6965    0.6125     16657

    accuracy                         0.5593     33314
   macro avg     0.5641    0.5593    0.5509     33314
weighted avg     0.5641    0.5593    0.5509     33314

Saved SVM, scaler, svm_results.json


In [ ]:
try:
    from xgboost import XGBClassifier
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'xgboost'])
    from xgboost import XGBClassifier

from sklearn.model_selection import GridSearchCV

rng = np.random.default_rng(SEED)
gi = rng.choice(len(X_train), min(MAX_GRID, len(X_train)), replace=False)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
}

print(f"Grid search on {len(gi)} samples...")
grid_xgb = GridSearchCV(
    XGBClassifier(random_state=SEED, eval_metric='logloss',
                  use_label_encoder=False),
    param_grid, cv=3, scoring='balanced_accuracy', n_jobs=1, verbose=1)
grid_xgb.fit(X_train[gi], y_train[gi])
print(f"Best: {grid_xgb.best_params_}  CV balanced acc {grid_xgb.best_score_:.4f}")

fi = rng.choice(len(X_train), min(MAX_FIT, len(X_train)), replace=False)
xgb_model = XGBClassifier(random_state=SEED, eval_metric='logloss',
                          use_label_encoder=False, **grid_xgb.best_params_)
xgb_model.fit(X_train[fi], y_train[fi])

test_acc_xgb = accuracy_score(y_test, xgb_model.predict(X_test))
print(f"\nXGBoost test accuracy: {test_acc_xgb:.4f}")
print(classification_report(y_test, xgb_model.predict(X_test),
                            target_names=['background', 'mass'], digits=4))

print(f"\n--- Comparison ---")
print(f"Nearest-centroid : 0.5340")
print(f"SVM (RBF)        : 0.5449")
print(f"XGBoost          : {test_acc_xgb:.4f}")

importances = xgb_model.feature_importances_
print(f"\nFeature importances (T0-T15 = histogram bins, last = intensity):")
for i, imp in enumerate(importances):
    label = f"T{i}" if i < 16 else "Intensity"
    print(f"  {label:<10} {imp:.4f}")

Grid search on 8000 samples...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best: {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 100}  CV balanced acc 0.5563

XGBoost test accuracy: 0.5321
              precision    recall  f1-score   support

  background     0.5513    0.3455    0.4248     16657
        mass     0.5234    0.7188    0.6057     16657

    accuracy                         0.5321     33314
   macro avg     0.5374    0.5321    0.5153     33314
weighted avg     0.5374    0.5321    0.5153     33314


--- Comparison ---
Nearest-centroid : 0.5340
SVM (RBF)        : 0.5449
XGBoost          : 0.5321

Feature importances (T0-T15 = histogram bins, last = intensity):
  T0         0.0335
  T1         0.0473
  T2         0.0513
  T3         0.0447
  T4         0.0417
  T5         0.0448
  T6         0.0491
  T7         0.0455
  T8         0.0533
  T9         0.0365
  T10        0.0405
  T11        0.0563
  T12        0.0441
  T13        0.0493
  T14        0.0

In [ ]:
# Layer 4 probability maps 
print("Predicting Layer 4 probability maps...\n")
layer4_maps = {}
for _, row in tqdm(final_df_loaded.iterrows(), total=len(final_df_loaded), desc="Layer 4"):
    iid = row['image_id']
    if iid in layer3_maps and iid in preprocessed_results:
        layer4_maps[iid] = {'prob_map': predict_prob_map(
            layer3_maps[iid]['hist_map'], preprocessed_results[iid]['roi'],
            svm, scaler), 'cls': row['class']}

np.savez_compressed(os.path.join(OUTPUTS_DIR, "layer4_prob_maps.npz"),
                    **{k: v['prob_map'] for k, v in layer4_maps.items()})
print(f"\nSaved {len(layer4_maps)} probability maps.")

Predicting Layer 4 probability maps...



Layer 4: 100%|██████████| 537/537 [1:21:28<00:00,  9.10s/it]



Saved 537 probability maps.


In [ ]:
# Verification
def verify_section911():
    checks = {
        "Layer 2 complete"    : len(layer2_maps) == len(final_df_loaded),
        "Layer 3 complete"    : len(layer3_maps) == len(final_df_loaded),
        #">2 templates used"   : (K_TEMPLATES - unused) > 2,
        "SVM trained"         : hasattr(svm, 'support_vectors_'),
        "test acc > chance"   : test_acc > 0.5,
        "prob maps complete"  : len(layer4_maps) == len(final_df_loaded),
        "prob in [0,1]"       : all(d['prob_map'].min() >= 0 and d['prob_map'].max() <= 1
                                    for d in layer4_maps.values()),
        "saved to Drive"      : os.path.exists(os.path.join(OUTPUTS_DIR, "layer4_prob_maps.npz")),
    }
    print("=" * 52); print("SECTIONS 9-11 VERIFICATION"); print("=" * 52)
    ok = True
    for k, v in checks.items():
        print(f"  {'OK ' if v else 'FAIL'}  {k}"); ok = ok and v
    print("=" * 52)
    print("Proceed to notebook 6." if ok else "Review failures.")
    return ok

verify_section911()

SECTIONS 9-11 VERIFICATION
  OK   Layer 2 complete
  OK   Layer 3 complete
  OK   SVM trained
  OK   test acc > chance
  OK   prob maps complete
  OK   prob in [0,1]
  OK   saved to Drive
Proceed to notebook 6.


True